# Capa Gold - Ventas

## Librerias

In [1]:
from pyspark.sql import functions as f
from pyspark.sql.types import DecimalType

## Variables

In [2]:
%run ../00-Modulos/n_modulo_variables.ipynb

Servicios:
 Warehouse  = s3a://iceberg/warehouse
 EndPoint   = http://localhost:9000
 Uri        = thrift://localhost:9083
Base de Datos Landing =  iceberg.landing
Base de Datos Bronze  =  iceberg.bronze
Base de Datos Silver  =  iceberg.silver
Base de Datos Gold    =  iceberg.gold


## Spark Session

In [3]:
spark = spark_session_hive("Lakehouse-Iceberg")

## Extracción

In [4]:
df_sales_transactions = spark.read.table(f'{vSchemaLand}.sales_transactions')
df_transactions = spark.read.table(f'{vSchemaSilv}.transactions')
df_dim_payments = spark.read.table(f'{vSchemaGold}.dim_payments')
df_dim_customers = spark.read.table(f'{vSchemaGold}.dim_customers')
df_dim_dealerships = spark.read.table(f'{vSchemaGold}.dim_dealerships')
df_dim_products = spark.read.table(f'{vSchemaGold}.dim_products')
df_dim_promotions = spark.read.table(f'{vSchemaGold}.dim_promotions')

## Transformaciones

In [5]:
# Fechas a procesar de la capa Landing
df_sales_transactions = df_sales_transactions \
    .select(f.date_format(f.col("TRANSACTION_DATE"), "yyyy-MM-dd").alias("TRANSACTION_DATE")) \
    .distinct()

# Obtener registros nuevos y calcula su llave subrogada
df_fact_sales = df_sales_transactions.alias("lt") \
    .join(df_transactions.alias("st"), (f.date_format(f.col("st.TRANSACTION_DATE"), "yyyy-MM-dd") == f.col("lt.TRANSACTION_DATE")), "inner") \
    .join(df_dim_payments.alias("p"), (f.col("st.PAYMENT_DESC") == f.col("p.PAYMENT_TYPE_DESC")), "inner") \
    .join(df_dim_customers.alias("c"), (f.col("st.CUST_ID") == f.col("c.CUST_ID")), "inner") \
    .join(df_dim_dealerships.alias("d"), (f.col("st.DEALERSHIP_ID") == f.col("d.DEALERSHIP_ID")), "inner") \
    .join(df_dim_products.alias("pd"), (f.col("st.PRODUCT_ID") == f.col("pd.PRODUCT_ID")), "inner") \
    .join(df_dim_promotions.alias("pm"), (f.col("st.PROMO_ID") == f.col("pm.PROMO_ID")), "inner") \
    .withColumn("DISCOUNT_F", ((f.when(f.col("st.DISCOUNT") > 17.25, f.col("pm.DISCOUNT")) \
        .otherwise(f.col("st.DISCOUNT"))) / 100) * f.col("st.SELLING_PRICE")) \
    .select(
        f.date_format(f.col("st.TRANSACTION_DATE"), "yyyyMMdd").cast("bigint").alias("DATE_SK"),
        f.col("c.CUST_SK"),
        f.col("pd.PRODUCT_SK"),
        f.col("d.DEALERSHIP_SK"),
        f.col("p.PAYMENT_SK"),
        f.col("pm.PROMO_SK"),  
        f.col("st.SALES_QTY").alias("UNITS_SOLD"),
        ((f.col("st.SELLING_PRICE") * f.col("st.SALES_QTY")) - f.col("DISCOUNT_F") - f.col("st.HOLDBACK") - f.col("st.REBATE")).alias("REVENUE"),
        f.col("st.UNIT_COST").alias("COST"),
        f.col("DISCOUNT_F").alias("DISCOUNT"),
        f.col("st.HOLDBACK"),
        f.col("st.REBATE")
    )

# Seleccion de columnas finales
df_source = df_fact_sales \
    .groupby(f.col("DATE_SK"), f.col("CUST_SK"), f.col("PRODUCT_SK"), f.col("DEALERSHIP_SK"), f.col("PAYMENT_SK"), f.col("PROMO_SK")) \
    .agg(
        f.sum(f.col("UNITS_SOLD")).alias("UNITS_SOLD"),
        f.sum(f.col("REVENUE")).cast(DecimalType(15, 2)).alias("REVENUE"),
        f.sum(f.col("COST")).cast(DecimalType(15, 2)).alias("COST"),
        f.avg(f.col("DISCOUNT")).cast(DecimalType(15, 2)).alias("DISCOUNT"),
        f.avg(f.col("HOLDBACK")).cast(DecimalType(15, 2)).alias("HOLDBACK"),
        f.avg(f.col("REBATE")).cast(DecimalType(15, 2)).alias("REBATE")
    )

## Carga

In [6]:
# Obtenemos variables para merge
target_table = f'{vSchemaGold}.fact_sales'
join_columns = ["CUST_SK", "PRODUCT_SK", "DEALERSHIP_SK", "PAYMENT_SK", "PROMO_SK"]

# Realiza el merge de la información
iceberg_upsert(df_source, target_table, join_columns, "DATE_SK")